# NB14d — Probability Distribution Diagnostic (Pure Investigation, kein Profile-Lock)

**Datum:** 2026-05-28
**Auslöser:** NB14c Run 1-3 lieferten widersprüchliche Ergebnisse:
- Run 1: hardcoded cutoff `0.4096` → 0 Trades
- Run 2: dyn cutoff (no floor) → cutoff `0.4022` → Cluster-Saturation → PF 1.01
- Run 3: dyn cutoff + floor `0.4070` + `deterministic=True` → 0 Trades

**Wir wissen nicht ob:**
- `deterministic=True` das Modell kaputt macht
- `0.4096` aus NB14b ein stochastischer Lucky-Run war
- Distribution generell instabil ist
- Cutoff-Mechanismus falsch ist

**Locked Rule (Nico, 2026-05-28):** Wichtige Zahlen (Premium PF, Cutoff, Trade-Count, Max-Proba, Sigs/Tag) brauchen MINDESTENS 3 reruns mit mean/std-Reporting. Single-Run-Entscheidungen sind ab jetzt verboten.

**Was dieses Notebook NICHT macht:**
- ❌ Keine Profile-Calibration
- ❌ Keine Filter-Tests
- ❌ Keine V1-Aussagen
- ❌ Keine ANN-Updates

**Was es macht (6 Diagnose-Sections):**
1. Distribution Summary (min/max/mean/std/quantiles/unique/top-20-cluster) für TRAIN/VAL/TEST
2. Histogram (full range + zoom 0.38–0.43)
3. Cluster Analysis (diskrete Bands, Saturation-Pattern)
4. Deterministic Toggle Diff (gleicher seed, deterministic on vs off)
5. Multi-Seed-Consistency (seeds [42, 1, 7] × deterministic [False, True] = 6 Runs)
6. Calibration Reliability Curve (predicted proba vs realized win-rate)
7. Verdict-Klassifikation (A/B/C/D) — basierend auf den Daten, nicht Hypothesen

**Output:** `/results/nb14d/diagnostic_full_snapshot_{date}.json` mit allem.
**Erwartete Laufzeit:** ~5-10 Min (6 Modell-Trainings + Diagnose-Funktionen).

## Section 0 — Setup + Module Sync

In [ ]:
import os, sys, subprocess, gc, json, importlib
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    subprocess.run(['rm', '-rf', '/tmp/pace-algo'])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/ecoNC/pace-algo.git',
                    '/tmp/pace-algo'], check=True)
    subprocess.run(f'mkdir -p {PROJECT_ROOT}/core/eval {PROJECT_ROOT}/core/analysis {PROJECT_ROOT}/core/router',
                   shell=True)
    subprocess.run(f'cp -rf /tmp/pace-algo/core/. {PROJECT_ROOT}/core/', shell=True)
    subprocess.run(f"find {PROJECT_ROOT}/core -type d -name __pycache__ -exec rm -rf {{}} +", shell=True)
    print('Core synced.')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
importlib.invalidate_caches()

import numpy as np
RUN_DATE      = datetime.now(timezone.utc).strftime('%Y-%m-%d')
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
try:
    GIT_COMMIT = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', '--short', 'HEAD'], text=True).strip()
except Exception:
    GIT_COMMIT = 'unknown'
EXPERIMENT_ID = f'nb14d_{RUN_TIMESTAMP}_{GIT_COMMIT}'
print(f'EXPERIMENT_ID: {EXPERIMENT_ID}')

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'lightgbm>=4.3', 'pyarrow>=15.0'], capture_output=True)

import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from core import config as cfg
from core.config import FX_TRAIN_SYMBOLS, FX_HOLDOUT_SYMBOLS, TRAIN_END, VAL_END
from core.train.dataset import walk_forward_split, binary_label_for_long, NON_FEATURE_COLS
from core.analysis.probability_diagnostic import (
    distribution_summary, histogram_data, find_discrete_clusters,
    calibration_curve_data, multi_seed_consistency, deterministic_toggle_diff,
    classify_verdict,
)

TF = '5m'
R_VALUE = 1.5
OUTPUT_DIR = Path(PROJECT_ROOT) / 'results' / 'nb14d'
for sub in ('metrics', 'summaries', 'config_snapshots'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'TF: {TF}  |  Train: {FX_TRAIN_SYMBOLS}  |  Hold-Out: {FX_HOLDOUT_SYMBOLS}')
print(f'Output: {OUTPUT_DIR}')

## Section 1 — Load Data + Walk-Forward Split

In [ ]:
if IN_COLAB:
    DATA_EXT = Path('/content/processed_v2')
    DATA_PROCESSED_LOCAL = Path('/content/processed')
    EXT_DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed_v2'
    DRIVE_BACKUP_PROCESSED = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    DATA_EXT.mkdir(parents=True, exist_ok=True)
    DATA_PROCESSED_LOCAL.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED_LOCAL.glob('labels_*.parquet'))) == 0 and DRIVE_BACKUP_PROCESSED.exists():
        subprocess.run(['rsync', '-ah', f'{DRIVE_BACKUP_PROCESSED}/', f'{DATA_PROCESSED_LOCAL}/'])
    if len(list(DATA_EXT.glob('*_extended.parquet'))) == 0 and EXT_DRIVE_BACKUP.exists():
        subprocess.run(['rsync', '-ah', f'{EXT_DRIVE_BACKUP}/', f'{DATA_EXT}/'])
else:
    DATA_EXT = cfg.DATA_PROCESSED.parent / 'processed_v2'
    DATA_PROCESSED_LOCAL = cfg.DATA_PROCESSED

def load_ext(sym, tf):
    p = DATA_EXT / f'{sym}_{tf}_extended.parquet'
    if not p.exists():
        return None
    return pd.read_parquet(p)

frames = []
for s in FX_TRAIN_SYMBOLS:
    d = load_ext(s, TF)
    if d is not None and not d.empty:
        d['symbol'] = s
        frames.append(d.astype({c: 'float32' for c in d.select_dtypes(include=['float64']).columns}))
pool = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

probe = load_ext(FX_TRAIN_SYMBOLS[0], TF)
FEATURE_COLS = [c for c in probe.columns if c not in NON_FEATURE_COLS and c != 'symbol']
print(f'Features: {len(FEATURE_COLS)}')

pool_c = pool.dropna(subset=FEATURE_COLS + ['label'])
train_df, val_df, test_df = walk_forward_split(pool_c, TRAIN_END, VAL_END)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = binary_label_for_long(train_df['label']).values
X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = binary_label_for_long(val_df['label']).values
X_test = test_df[FEATURE_COLS].values.astype(np.float32)
y_test = binary_label_for_long(test_df['label']).values
y_test_triple = test_df['label'].values
del pool, pool_c, train_df, val_df, test_df
gc.collect()
print('Data loaded + features extracted.')

## Section 2 — Helpers (LightGBM Train + Predict)

In [ ]:
# Lokale train-Funktion mit konfigurierbarem seed/deterministic
# (statt core.train.lgbm_trainer.train_lgbm — wir wollen volle Kontrolle für Diagnostik)
BASE_PARAMS = {
    'objective':            'binary',
    'metric':               'binary_logloss',
    'num_leaves':           7,
    'max_depth':            3,
    'min_data_in_leaf':     200,
    'learning_rate':        0.05,
    'num_iterations':       100,
    'lambda_l2':            1.0,
    'feature_fraction':     0.8,
    'bagging_fraction':     0.8,
    'bagging_freq':         5,
    'is_unbalance':         True,
    'verbose':              -1,
    'n_jobs':               -1,
}

def train_with_params(X_tr, y_tr, X_vl, y_vl, params):
    """Train LightGBM mit custom params — für Multi-Seed-Diagnostik."""
    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    model = lgb.train(
        params, td,
        valid_sets=[td, vd],
        valid_names=['train', 'val'],
        callbacks=[lgb.early_stopping(10, verbose=False), lgb.log_evaluation(period=0)],
    )
    return model

def predict_proba(model, X):
    return model.predict(X)

print('Helpers defined.')

## Section 3 — Baseline-Run (seed=42, deterministic=False)

Erste Diagnose: Trainiere mit seed=42 ohne deterministic. Dann analysiere proba_train / proba_val / proba_test.

In [ ]:
baseline_params = dict(BASE_PARAMS)
baseline_params['seed'] = 42
baseline_params['deterministic'] = False

model_baseline = train_with_params(X_train, y_train, X_val, y_val, baseline_params)
p_train_baseline = predict_proba(model_baseline, X_train)
p_val_baseline   = predict_proba(model_baseline, X_val)
p_test_baseline  = predict_proba(model_baseline, X_test)

print(f'Train iter: {model_baseline.best_iteration}/{baseline_params["num_iterations"]}')
print(f'TRAIN proba: min={p_train_baseline.min():.4f}, max={p_train_baseline.max():.4f}, mean={p_train_baseline.mean():.4f}')
print(f'VAL   proba: min={p_val_baseline.min():.4f}, max={p_val_baseline.max():.4f}, mean={p_val_baseline.mean():.4f}')
print(f'TEST  proba: min={p_test_baseline.min():.4f}, max={p_test_baseline.max():.4f}, mean={p_test_baseline.mean():.4f}')

# Distribution Summaries
summary_train = distribution_summary(p_train_baseline, 'TRAIN_baseline_seed42_detFalse')
summary_val   = distribution_summary(p_val_baseline,   'VAL_baseline_seed42_detFalse')
summary_test  = distribution_summary(p_test_baseline,  'TEST_baseline_seed42_detFalse')

print(f'\n=== VAL Top-20 unique values (rounded to 4dp) ===')
for v in summary_val['top20_values'][:10]:
    print(f'  {v["value"]:.4f}  count={v["count"]:>6}  ({v["pct_of_total"]:.2f}%)')

print(f'\n=== Counts at typical cutoff thresholds (TEST) ===')
for k, v in summary_test['counts_at_cutoffs'].items():
    print(f'  >={k}:  {v["n_bars_above"]:>6} bars ({v["pct_above"]:.2f}%)')

baseline_summaries = {
    'train': summary_train,
    'val':   summary_val,
    'test':  summary_test,
}

## Section 4 — Histogram Data (Full + Zoom 0.38–0.43)

In [ ]:
histograms = {
    'val_full':  histogram_data(p_val_baseline,  bins=80, zoom_range=(0.38, 0.43)),
    'test_full': histogram_data(p_test_baseline, bins=80, zoom_range=(0.38, 0.43)),
}

# Print text-summary der Zoom-Range (relevant für Cluster-Analyse)
print('=== VAL histogram zoom 0.38-0.43 ===')
z = histograms['val_full']['zoom_range']
print(f'n_in_range: {z["n_in_range"]}')
if z['n_in_range'] > 0:
    print('edge_low → edge_high : count')
    for i in range(len(z['counts'])):
        if z['counts'][i] > 0:
            print(f'  {z["edges"][i]:.4f} → {z["edges"][i+1]:.4f} : {z["counts"][i]}')

print('\n=== TEST histogram zoom 0.38-0.43 ===')
z = histograms['test_full']['zoom_range']
print(f'n_in_range: {z["n_in_range"]}')
if z['n_in_range'] > 0:
    print('edge_low → edge_high : count')
    for i in range(len(z['counts'])):
        if z['counts'][i] > 0:
            print(f'  {z["edges"][i]:.4f} → {z["edges"][i+1]:.4f} : {z["counts"][i]}')

## Section 5 — Cluster Analysis

Wie diskret ist die Probability-Verteilung? Wenn Top-3-Cluster > 50% aller Bars → ANN-012's Annahme bestätigt.

In [ ]:
cluster_val  = find_discrete_clusters(p_val_baseline,  decimal_places=4, min_cluster_size=10)
cluster_test = find_discrete_clusters(p_test_baseline, decimal_places=4, min_cluster_size=10)

print('=== VAL Cluster-Analyse (rounded to 4dp) ===')
print(f'Total bars: {cluster_val["n_total_bars"]:,}')
print(f'Distinct cluster: {cluster_val["n_clusters"]}')
print(f'Top-3-Cluster sum: {cluster_val["top3_cluster_pct"]:.2f}%')
print(f'is_highly_discrete: {cluster_val["is_highly_discrete"]}')
print('\nTop-10 clusters:')
for c in cluster_val['clusters'][:10]:
    print(f'  value={c["value"]:.4f}  count={c["count"]:>6}  ({c["pct"]:.2f}%)')

print('\n=== TEST Cluster-Analyse ===')
print(f'Top-3 cluster sum: {cluster_test["top3_cluster_pct"]:.2f}%')
print(f'is_highly_discrete: {cluster_test["is_highly_discrete"]}')
print('Top-10 clusters:')
for c in cluster_test['clusters'][:10]:
    print(f'  value={c["value"]:.4f}  count={c["count"]:>6}  ({c["pct"]:.2f}%)')

## Section 6 — Deterministic Toggle Diff (Critical Test)

Gleicher seed (42), deterministic=True vs False. Verändert das Flag das Modell signifikant?

In [ ]:
det_true_params = dict(BASE_PARAMS)
det_true_params['seed'] = 42
det_true_params['deterministic'] = True

model_det = train_with_params(X_train, y_train, X_val, y_val, det_true_params)
p_val_det   = predict_proba(model_det, X_val)
p_test_det  = predict_proba(model_det, X_test)

diff_val  = deterministic_toggle_diff(p_val_baseline,  p_val_det)
diff_test = deterministic_toggle_diff(p_test_baseline, p_test_det)

print('=== Deterministic Toggle Diff — VAL ===')
for k, v in diff_val.items():
    if isinstance(v, float):
        print(f'  {k:25s} {v:.6f}')
    else:
        print(f'  {k:25s} {v}')

print('\n=== Deterministic Toggle Diff — TEST ===')
for k, v in diff_test.items():
    if isinstance(v, float):
        print(f'  {k:25s} {v:.6f}')
    else:
        print(f'  {k:25s} {v}')

## Section 7 — Multi-Seed Consistency (6 Runs: seeds×det)

Seeds [42, 1, 7] × deterministic [False, True] = 6 Runs. Misst: wie stark schwankt der Cutoff?

In [ ]:
def train_fn(X_tr, y_tr, X_vl, y_vl, params):
    return train_with_params(X_tr, y_tr, X_vl, y_vl, params)

def predict_fn(model, X):
    return predict_proba(model, X)

multi_seed = multi_seed_consistency(
    train_fn=train_fn,
    predict_fn=predict_fn,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,
    base_params=BASE_PARAMS,
    seeds=[42, 1, 7],
    deterministic_options=[False, True],
)

print('=== Multi-Seed Consistency Aggregates ===')
for k, v in multi_seed['aggregates'].items():
    print(f'  {k:30s} {v:.4f}' if isinstance(v, float) else f'  {k:30s} {v}')

print(f'\ncutoff_drift:       {multi_seed["cutoff_drift"]:.4f}  (max-min across {len(multi_seed["runs"])} runs)')
print(f'max_proba_drift:    {multi_seed["max_proba_drift"]:.4f}')
print(f'stable_enough:      {multi_seed["stable_enough"]}')

print('\n=== Per-Run Breakdown ===')
for r in multi_seed['runs']:
    print(f'  seed={r["seed"]:>3}  det={str(r["deterministic"]):>5}  '
          f'cutoff_q99={r["cutoff_top1pct"]:.4f}  '
          f'test_max={r["proba_test_max"]:.4f}  '
          f'n_above_q99={r["n_test_above_q99"]:>4}  '
          f'n_above_0.4096={r["n_test_above_0_4096"]:>4}  '
          f'n_above_0.4070={r["n_test_above_0_4070"]:>4}')

## Section 8 — Calibration Reliability Curve

Predicted Probability vs Realized Win-Rate. ECE > 0.10 = Modell-Calibration kaputt.

In [ ]:
calib_val  = calibration_curve_data(y_val,  p_val_baseline,  n_bins=10)
calib_test = calibration_curve_data(y_test, p_test_baseline, n_bins=10)

print('=== VAL Calibration ===')
print(f'ECE: {calib_val["ece"]:.4f}  ({calib_val["ece_interpretation"]})')
print('bin  edge_low  edge_high  n     predicted  actual    gap')
for b in calib_val['bins']:
    print(f'  {b["bin"]}  {b["edge_low"]:.4f}    {b["edge_high"]:.4f}    {b["n"]:>5}  '
          f'{b["predicted_mean"]:.4f}    {b["actual_positive"]:.4f}    {b["calibration_gap"]:+.4f}')

print(f'\n=== TEST Calibration ===')
print(f'ECE: {calib_test["ece"]:.4f}')
print('bin  edge_low  edge_high  n     predicted  actual    gap')
for b in calib_test['bins']:
    print(f'  {b["bin"]}  {b["edge_low"]:.4f}    {b["edge_high"]:.4f}    {b["n"]:>5}  '
          f'{b["predicted_mean"]:.4f}    {b["actual_positive"]:.4f}    {b["calibration_gap"]:+.4f}')

## Section 9 — Verdict-Klassifikation (A/B/C/D)

Automatische Klassifikation basierend auf den Daten.

In [ ]:
verdict = classify_verdict(
    multi_seed_result=multi_seed,
    det_toggle_diff=diff_val,
    cluster_result=cluster_val,
    calibration_result=calib_val,
)

print('=' * 70)
print(f'NB14d VERDICT — {RUN_DATE}')
print('=' * 70)
print('\nFindings:')
for f in verdict['findings']:
    print(f'  • {f}')
print('\nRecommendations:')
for r in verdict['recommendations']:
    print(f'  → {r}')
print('=' * 70)

## Section 10 — Persistence (Full JSON Snapshot)

In [ ]:
snapshot = {
    'experiment_id':   EXPERIMENT_ID,
    'run_date':        RUN_DATE,
    'git_commit':      GIT_COMMIT,
    'tf':              TF,
    'fx_train_symbols': FX_TRAIN_SYMBOLS,
    'fx_holdout_symbols': FX_HOLDOUT_SYMBOLS,
    'n_train':         int(len(X_train)),
    'n_val':           int(len(X_val)),
    'n_test':          int(len(X_test)),
    'baseline_summaries': baseline_summaries,
    'histograms':         histograms,
    'cluster_val':        cluster_val,
    'cluster_test':       cluster_test,
    'deterministic_toggle_val':  diff_val,
    'deterministic_toggle_test': diff_test,
    'multi_seed_consistency':    multi_seed,
    'calibration_val':           calib_val,
    'calibration_test':          calib_test,
    'verdict':                   verdict,
}

snap_path = OUTPUT_DIR / 'summaries' / f'nb14d_diagnostic_full_{RUN_DATE}.json'
with open(snap_path, 'w') as f:
    json.dump(snapshot, f, indent=2, default=str)
print(f'Snapshot → {snap_path}')
print(f'Size: {snap_path.stat().st_size / 1024:.1f} KB')

# Config snapshot mit Run-Parametern
config_snap = {
    'experiment_id': EXPERIMENT_ID,
    'run_date': RUN_DATE,
    'git_commit': GIT_COMMIT,
    'base_params': BASE_PARAMS,
    'seeds_tested': [42, 1, 7],
    'deterministic_tested': [False, True],
    'features': len(FEATURE_COLS),
    'n_runs_total': len(multi_seed['runs']),
}
cfg_path = OUTPUT_DIR / 'config_snapshots' / f'{EXPERIMENT_ID}_config.json'
with open(cfg_path, 'w') as f:
    json.dump(config_snap, f, indent=2, default=str)
print(f'Config → {cfg_path}')

## Section 11 — Auto-Push to GitHub

In [ ]:
import shutil
if not IN_COLAB:
    print('Local run — skip push.')
else:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception as e:
        GH_TOKEN = None
        print(f'ERROR cannot read GITHUB_TOKEN: {e}')

    if GH_TOKEN:
        NB_TAG = 'nb14d'
        TMP_REPO = Path('/tmp/pace-algo-push')
        if TMP_REPO.exists():
            shutil.rmtree(TMP_REPO)
        subprocess.run(['git', 'clone', '--quiet',
                        f'https://{GH_TOKEN}@github.com/ecoNC/pace-algo.git', str(TMP_REPO)], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.name', 'ecoNC'], check=True)
        subprocess.run(['git', '-C', str(TMP_REPO), 'config', 'user.email',
                        'ecoNC@users.noreply.github.com'], check=True)

        # Pull --rebase ZUERST (Lesson aus NB14c Run 2/3)
        subprocess.run(['git', '-C', str(TMP_REPO), 'pull', '--rebase', '--autostash', '--quiet',
                        'origin', 'main'], check=True)
        print('Pulled latest from origin/main')

        # Dann Files kopieren
        copied = []
        for f in sorted(OUTPUT_DIR.rglob(f'*{RUN_DATE}*')):
            if not f.is_file(): continue
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        for f in sorted((OUTPUT_DIR / 'config_snapshots').glob(f'*{EXPERIMENT_ID}*')):
            rel = f.relative_to(Path(PROJECT_ROOT) / 'results')
            dest = TMP_REPO / 'results' / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, dest)
            copied.append(rel)
        print(f'Copied {len(copied)} files')

        subprocess.run(['git', '-C', str(TMP_REPO), 'add', 'results/'], check=True)
        r_status = subprocess.run(['git', '-C', str(TMP_REPO), 'status', '--porcelain'],
                                    capture_output=True, text=True)
        if not r_status.stdout.strip():
            print('Nothing to commit.')
        else:
            msg = f'{NB_TAG.upper()} probability distribution diagnostic {RUN_DATE} ({len(copied)} files)'
            subprocess.run(['git', '-C', str(TMP_REPO), 'commit', '-m', msg], check=True)
            sha = subprocess.run(['git', '-C', str(TMP_REPO), 'rev-parse', '--short', 'HEAD'],
                                   capture_output=True, text=True).stdout.strip()
            subprocess.run(['git', '-C', str(TMP_REPO), 'push', 'origin', 'main'], check=True)
            print(f'✓ Pushed as ecoNC ({sha})')
            print(f'  https://github.com/ecoNC/pace-algo/commit/{sha}')
        shutil.rmtree(TMP_REPO)